# Μάθημα 08 - Πρότυπο Σχεδίασης Πολλαπλών Πρακτόρων


## Ρύθμιση


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Γιατί Πολλαπλά Συστήματα Πρακτόρων;

Οι εργασίες του πραγματικού κόσμου, όπως ο προγραμματισμός ταξιδιού, περιλαμβάνουν πολλούς διαφορετικούς τύπους εμπειρογνωμοσύνης — logistics, τοπική γνώση, προϋπολογισμό και άλλα. Ένας μόνος πράκτορας που προσπαθεί να χειριστεί τα πάντα γρήγορα γίνεται δύσχρηστος.

Τα συστήματα πολλαπλών πρακτόρων λύνουν αυτό το πρόβλημα μέσω της **εξειδίκευσης**: κάθε πράκτορας επικεντρώνεται σε έναν τομέα ειδίκευσης, παράγοντας αποτελέσματα υψηλότερης ποιότητας από έναν γενικευτή. Επίσης βελτιώνουν την **επεκτασιμότητα** — μπορείτε να προσθέτετε νέους πράκτορες (π.χ., έναν ειδικό πτήσεων, έναν κριτικό εστιατορίων) χωρίς να ξαναγράφετε τη υπάρχουσα ροή εργασίας. Οι πράκτορες συνθέτονται μαζί μέσω μιας δομημένης αλυσίδας, περνώντας πλαίσιο από τον έναν στον επόμενο.


## Δημιουργία Εξειδικευμένων Πρακτόρων


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Δημιουργία μιας Ακολουθιακής Ροής Εργασίας

Το `WorkflowBuilder` σας επιτρέπει να συνδέσετε πράκτορες σε έναν κατευθυνόμενο γράφο. Εδώ δημιουργούμε μια απλή διβήματη ροή εργασίας: ο **TravelPlanner** σχεδιάζει το δρομολόγιο και στη συνέχεια ο **TravelConcierge** το εξετάζει και το βελτιώνει.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Προσθήκη Περισσότερων Πρακτόρων στη Ροή Εργασίας

Ένα από τα μεγαλύτερα πλεονεκτήματα του μοτίβου πολλαπλών πρακτόρων είναι πόσο εύκολα μπορεί να επεκταθεί. Παρακάτω προσθέτουμε έναν πράκτορα **BudgetReviewer** που ελέγχει το σχέδιο σε σχέση με τον προϋπολογισμό του ταξιδιώτη, επισημαίνει στοιχεία που μπορεί να αυξήσουν τα έξοδα πέρα από το όριο και προτείνει εναλλακτικές που εξοικονομούν χρήματα. Η ροή εργασίας τώρα τρέχει τρεις πράκτορες κατά σειρά:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Περίληψη

Σε αυτό το μάθημα μάθατε πώς να:

1. **Δημιουργείτε εξειδικευμένους πράκτορες** — ο καθένας με συγκεκριμένο ρόλο (σχεδιασμός, κονσιέρζ, ανασκόπηση προϋπολογισμού).
2. **Συνδέετε τους πράκτορες σε μια αλληλουχία εργασίας** χρησιμοποιώντας `WorkflowBuilder` και `add_edge`.
3. **Ρεύμα εξόδου** από μια αλυσίδα πολλαπλών πρακτόρων, παρακολουθώντας ποιος πράκτορας μιλά.
4. **Επεκτείνετε μια εργασία** προσθέτοντας νέους πράκτορες στην αλυσίδα χωρίς να τροποποιήσετε τους υπάρχοντες.

Το πρότυπο σχεδίασης πολλαπλών πρακτόρων διατηρεί κάθε πράκτορα απλό, ενώ παράγει πλουσιότερα, πιο ελεγμένα αποτελέσματα από ό,τι θα μπορούσε να πετύχει μόνος του ένας πράκτορας.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία αυτόματης μετάφρασης AI [Co-op Translator](https://github.com/Azure/co-op-translator). Παρόλο που καταβάλλουμε προσπάθειες για ακρίβεια, παρακαλούμε να γνωρίζετε ότι οι αυτόματες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα θα πρέπει να θεωρείται η επίσημη πηγή. Για κρίσιμες πληροφορίες, συνιστάται η επαγγελματική μετάφραση από άνθρωπο. Δεν φέρουμε ευθύνη για οποιεσδήποτε παρεξηγήσεις ή λανθασμένες ερμηνείες προκύψουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
